# Lab Assignment 5: Restoring Clean Images from Noisy Images

## Cleaning Up the Bayes' Theorem Image
Now it's your turn! It's time to design an implement your solution to clean up `./figures/Bayes-noise.png`. Use the pre-processed `./figures/Bayes-pre-processed.png`, provided. Be careful not to over-write these files. Save your cleaned up image with a different file name (for example, `./figures/Bayes-denoised.png`) to display and check your work. **DO NOT use the cautionary tale image, `./bad_figures/Bayes-incorrectly-pre-processed.png` as your starting point.**

You are welcome to copy/paste any of the starter code above in your solution.

The problem statement above is copied here for your convenience.

### Revisiting the Problem Statement:
![A clean image, the corrupted image, and a pairwise MRF](./figures/problem_statement_diagram.png)

The goal of this problem is to recover the clean image (a) from a noisy input (b). Of course, we cannot recover (a) exactly since information has been lost in the noise. The graphical model we use is a pairwise MRF as shown in (c). A pixel has four neighbors. We are going to use Markov Random Fields (MRFs) to model the distribution of natural images and restore the clean image given an noisy input image. Observed image $y_i \in \{-1,+1\}$, and $i = 1,\cdots,D$ indexes pixels in the lattice. The
original noise free image is $x_i \in \{-1, +1\}$. The noisy image $y_i$ is obtained by randomly flipping the sign of pixels with some probability (10% of pixels, in this case).

There are two types of cliques in this MRF. For $\{x_i,y_i\}$, we define the energy function as $-\eta x_i y_i\ (\eta > 0)$.  Lower energy is achieved when $x_i$ and $y_i$ have the same sign and a higher energy when they have the opposite sign. For a pair of variables $\{x_i, x_j\}$ where $i$ and $j$ are indices of neighboring pixels, we want the energy to be lower when they have the same sign than when they have opposite sign. So the energy function is $-\beta x_i j_i\ (\beta > 0)$. Lastly, we have an energy term $hx_i$ for each pixel $i$ to bias the model towards one particular sign (either $+$ or $−$).

The final energy function for the model takes the form

$E(\vec{x}\vec{y}) = h\displaystyle\sum\limits_i x_i - \beta \displaystyle\sum\limits_{\{i,j\}} x_ix_j - \eta\displaystyle\sum\limits_i x_iy_i$

which defines a joint distribution over $\vec{x}$ and $\vec{y}$ given by

$p(\vec{x},\vec{y}) = \dfrac{1}{\vec{\mathcal{Z}}}exp\{-E(\vec{x}\vec{y})\}$

Now implement Coordinate-descent algorithm as below on this:
1. Initialize $\{x_i\} (x_i = y_i)$
2. Loop over $\{x_i\}$. For each $x_i$, fix the neighborhood and see whether $−x_i$ would
decrease the energy. If so, then flip $x_i$; otherwise, continue.
3. Stop when no changes can be made for $x$.

Now make some initial guess for the parameters $h$, $\beta$, $\eta$ so that the above algorithm converges and then adjust them until you can get up to 96% recovery or better. Record your parameter values and the number of iterations when you acheived this recovery.

Save the recovered image and submit it along with your code. The clean image, noisy image, and now the pre-processed images are included in the `figures` folder : `./figures/Bayes.png`, `./figures/Bayes-noise.png`, and `./figures/Bayes-pre-processed.png`

### Write and Run your Own Code

## Import Libraries
In addition to the `IPython.display` and the `PIL` packages, if you would like to use other libraries in this notebook, please include them below.

In [5]:
## ===============================================================
## Library Imports:
## ===============================================================
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
## Image display functionality
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
from IPython.display import Image as disp

## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
## Pixel Manipulation
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
from PIL import Image

## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
## Additional Libraries?
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#### Please insert other libraries as needed.

#### I recommend to use these as well; feel free to use different / remove if unnecessary
import numpy as np
import matplotlib.pyplot as plt
import copy
import os

In [6]:
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
## Load images and convert to black and white
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
clean_arr = np.array(Image.open('./figures/Bayes.png').convert(mode='1', dither=0), dtype=np.uint8)
noisy_arr = np.array(Image.open('./figures/Bayes-pre-processed.png').convert(mode='1', dither=0), dtype=np.uint8)

def to_spin(arr):
    return np.where(arr > 0, np.int8(1), np.int8(-1))

x_clean = to_spin(clean_arr)
y_obs   = to_spin(noisy_arr)

H, W = y_obs.shape
print(f'Image: {W}x{H} pixels')
print(f'Baseline noisy accuracy: {(y_obs == x_clean).mean()*100:.2f}%')

In [ ]:
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
## Implement your coordinate-descent algorithm here
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

def four_neighbor_sum(x):
    s = np.zeros_like(x, dtype=np.int16)
    s[:-1, :] += x[1:, :]
    s[1:,  :] += x[:-1, :]
    s[:, :-1] += x[:, 1:]
    s[:,  1:] += x[:, :-1]
    return s

def mrf_energy(x, y, h, beta, eta):
    nbr = four_neighbor_sum(x)
    return (h * float(x.sum())
            - 0.5 * beta * float((x * nbr).sum())
            - eta  * float((x * y).sum()))

def run_coordinate_descent(y_init, y_obs, h, beta, eta, max_iter=30):
    x = y_init.copy().astype(np.int8)
    rows, cols = np.indices(x.shape)
    even_mask = (rows + cols) % 2 == 0

    history = []
    for it in range(1, max_iter + 1):
        n_flips = 0
        for mask in (even_mask, ~even_mask):
            field  = beta * four_neighbor_sum(x) + eta * y_obs - h
            target = np.where(field > 0, np.int8(1), np.where(field < 0, np.int8(-1), x))
            flip   = mask & (target != x)
            n_flips += int(flip.sum())
            x[flip] = target[flip]
        energy = mrf_energy(x, y_obs, h, beta, eta)
        history.append((it, n_flips, energy))
        print(f'Iter {it:2d}: flips={n_flips:6d},  energy={energy:12.1f}')
        if n_flips == 0:
            break
    return x, history

h, beta, eta = 0.0, 1.0, 1.5
print(f'Parameters: h={h}, beta={beta}, eta={eta}\n')
x_est, run_history = run_coordinate_descent(y_obs, y_obs, h=h, beta=beta, eta=eta, max_iter=30)

final_acc = (x_est == x_clean).mean() * 100
print(f'\nConverged in {len(run_history)} iteration(s)')
print(f'Final recovery accuracy: {final_acc:.2f}%')

In [ ]:
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
## Display and save your result
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

out_px = np.where(x_est > 0, np.uint8(255), np.uint8(0))
Image.fromarray(out_px, mode='L').save('./figures/Bayes-denoised.png')

panels = [
    (np.where(clean_arr > 0, 255, 0), 'Clean (reference)'),
    (np.where(noisy_arr > 0, 255, 0), 'Noisy input'),
    (out_px,                          f'Denoised  ({final_acc:.2f}%)'),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (arr, title) in zip(axes, panels):
    ax.imshow(arr, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig('./figures/comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved to ./figures/Bayes-denoised.png')
disp('./figures/Bayes-denoised.png')

### What to Submit
Please submit the following:

1. A brief post-lab write-up that contains the following for this assignment:

    a. Any paper design that you have.
    
    b. A brief description of your model/algorithm. Justify your design/selection of model parameters as appropriate.
    
    c. An evaluation of your model, including evidence as appropriate.
    
    d. A brief (couple of sentences) reflection on your take-aways from this lab exercise.

In [ ]:
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
## Tests -- run BEFORE the algorithm cells above to catch bugs early
## +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

def run_tests():
    passed = 0; failed = 0
    def check(name, ok, note=''):
        nonlocal passed, failed
        if ok:
            print(f'  PASS  {name}')
            passed += 1
        else:
            print(f'  FAIL  {name}  {note}')
            failed += 1

    # ── four_neighbor_sum ──
    print('--- four_neighbor_sum ---')
    z = np.zeros((5, 5), dtype=np.int8)
    z[2, 2] = 1
    ns = four_neighbor_sum(z)
    check('point source propagates to 4 cardinal neighbors',
          ns[1,2]==1 and ns[3,2]==1 and ns[2,1]==1 and ns[2,3]==1)
    check('source pixel itself is zero', ns[2,2] == 0)
    check('corner pixel untouched',      ns[0,0] == 0)

    uni = np.ones((6, 6), dtype=np.int8)
    ns2 = four_neighbor_sum(uni)
    check('corner:   2 neighbors', ns2[0,0] == 2)
    check('edge:     3 neighbors', ns2[0,3] == 3)
    check('interior: 4 neighbors', ns2[2,2] == 4)

    # ── mrf_energy ──
    print('--- mrf_energy ---')
    x_agree  =  np.ones((4, 4), dtype=np.int8)
    x_oppose = -np.ones((4, 4), dtype=np.int8)
    y_all1   =  np.ones((4, 4), dtype=np.int8)
    check('x=y lower energy than x=-y',
          mrf_energy(x_agree, y_all1, 0.0, 1.0, 1.0) <
          mrf_energy(x_oppose, y_all1, 0.0, 1.0, 1.0))

    x_anti = np.array([[1,-1],[-1,1]], dtype=np.int8)
    x_uni2 = np.ones((2, 2), dtype=np.int8)
    y2     = np.ones((2, 2), dtype=np.int8)
    check('checkerboard has higher pair energy than uniform',
          mrf_energy(x_anti, y2, 0.0, 1.0, 0.0) >
          mrf_energy(x_uni2, y2, 0.0, 1.0, 0.0))

    # ── run_coordinate_descent ──
    print('--- run_coordinate_descent ---')

    x_uni3 = np.ones((8, 8), dtype=np.int8)
    x_r, h_r = run_coordinate_descent(x_uni3, x_uni3, h=0.0, beta=1.0, eta=1.5, max_iter=5)
    check('uniform image: no pixels change',         (x_r == 1).all())
    check('uniform image: exits after 1 iteration',  len(h_r)==1 and h_r[0][1]==0)

    # Structured image with large smooth regions -- MRF is designed for this, not random images
    x_true = np.ones((40, 40), dtype=np.int8)
    x_true[20:, :] = -1
    x_true[:, 20:] *= -1
    rng = np.random.default_rng(42)
    y_syn  = x_true.copy()
    y_syn[rng.random((40, 40)) < 0.10] *= -1
    x_rec, hist = run_coordinate_descent(y_syn, y_syn, h=0.0, beta=1.0, eta=1.5, max_iter=25)

    energies = [e for _, _, e in hist]
    monotone = all(energies[i] <= energies[i-1] + 1e-6 for i in range(1, len(energies)))
    check('energy non-increasing across iterations', monotone)
    check('algorithm terminates with 0 flips',       hist[-1][1] == 0)

    acc_syn = (x_rec == x_true).mean()
    check(f'synthetic 10%-noise accuracy >= 90%  (got {acc_syn:.1%})', acc_syn >= 0.90)

    x_lone    = np.ones((5, 5), dtype=np.int8)
    x_lone[2, 2] = -1
    x_fixed, _ = run_coordinate_descent(x_lone, x_lone, h=0.0, beta=1.0, eta=1.5, max_iter=5)
    check('lone noisy pixel on uniform background is corrected', x_fixed[2, 2] == 1)

    print(f'\n{passed} passed, {failed} failed')

run_tests()